# Structure and calibration checks

Use this notebook when a scan looks wrong. It isolates geometry and calibration from model reconstruction.

**Before running:** install `.[dev,notebooks]`, open Jupyter from the repository root, and run the setup cell first.

**Result:** a film-coordinate overlay, selected frequency and height breakpoints, and a calibrated view labelled in MHz and kilometres.

In [ ]:
import sys
from pathlib import Path

import matplotlib

_interactive = False
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
    _interactive = 'agg' not in str(matplotlib.get_backend()).lower()
except NameError:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt


def show_plot():
    if _interactive and 'agg' not in str(matplotlib.get_backend()).lower():
        plt.show()
    plt.close()

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / 'pyproject.toml').is_file():
        ROOT = candidate
        break
else:
    raise RuntimeError('Open this notebook from the repository root')
sys.path[:0] = [str(ROOT), str(ROOT / 'src')]
film_path = ROOT / 'data/raw/csa_verified_bur_1973077231124.png'
profile_path = ROOT / 'configs/film_calibration_profile.json'
assert film_path.is_file(), 'The matched CSA sample is missing from data/raw/'

from isis_research.image_io import load_image
from scripts.pipeline.extract_scan_structure import extract_structure, write_overlay
from scripts.pipeline.fit_frequency_axis import fit_from_profile, load_json
from scripts.pipeline.fit_height_axis import fit_from_profile as fit_height
from scripts.pipeline.standardize_film_only_512 import collapse_duplicate_fallback
from scripts.pipeline.warp_calibrated_scan import warp_one

image = load_image(film_path)
structure = extract_structure(image)
profile = load_json(profile_path)
frequency = collapse_duplicate_fallback(fit_from_profile([item['x'] for item in structure['vertical_markers']['candidates']], image.shape, profile))
height = fit_height(structure, profile, frequency)
print('Structure:', structure['status'], structure['warnings'])
print('Frequency:', frequency['status'], frequency.get('warnings', []))
print('Virtual height:', height['status'], height.get('warnings', []))

## What to inspect

Look for boundaries around the exposed ionogram, marker candidates on the printed vertical lines, and a regular ruling lattice. A low marker count, irregular lattice, or edge-touching film region should remain a review signal.

In [ ]:
plot_structure = dict(structure)
plot_structure['film_region'] = dict(structure['film_region'])
plot_path = ROOT / 'outputs/notebooks/02_calibration_overlay.png'
plot_path.parent.mkdir(parents=True, exist_ok=True)
write_overlay(plot_path, image, plot_structure, film_path.name)
plt.figure(figsize=(12, 7))
plt.imshow(plt.imread(plot_path))
plt.axis('off')
show_plot()

In [ ]:
print('Frequency breakpoints:')
for point in frequency.get('breakpoints', []):
    print(point)
print('Virtual-height breakpoints:')
for point in height.get('breakpoints', []):
    print(point)

result, arrays = warp_one(image, frequency, height, structure, frequency_bins=512, height_bins=512)
if arrays is None:
    raise RuntimeError(f'Calibration did not produce a calibrated view: {result["status"]} — {result.get("reason", "see warnings")}')
plt.figure(figsize=(12, 7))
plt.imshow(
    arrays['warped'],
    cmap='gray',
    aspect='auto',
    vmin=0,
    vmax=1,
    origin='upper',
    extent=[float(arrays['frequency'][0]), float(arrays['frequency'][-1]), float(arrays['height'][-1]), float(arrays['height'][0])],
)
plt.title('Calibrated view for axis inspection')
plt.xlabel('frequency (MHz)')
plt.ylabel('virtual height (km)')
show_plot()